# CardioIA — Fase 2 — Parte 2
## Classificação de risco em textos médicos com TF-IDF + Regressão Logística

O dataset é **simulado e exclusivamente acadêmico**. Os rótulos não devem ser usados para tomada de decisão clínica real.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
ARQ_BASE = ROOT / 'data' / 'base_risco.csv'
print('Base localizada em:', ARQ_BASE.resolve())

Base localizada em: /mnt/data/CardioIA_Fase2/data/base_risco.csv


### 1. Leitura e inspeção do dataset

In [2]:
df = pd.read_csv(ARQ_BASE)
print('Dimensão:', df.shape)
print('\nDistribuição das classes:')
print(df['rotulo'].value_counts())
df.head(10)

Dimensão: (120, 2)

Distribuição das classes:
rotulo
alto risco     60
baixo risco    60
Name: count, dtype: int64


,frase,rotulo
0,Paciente apresenta dor intensa no peito com co...,alto risco
1,Paciente apresenta aperto torácico discreto le...,baixo risco
2,Paciente apresenta mal-estar passageiro leve a...,baixo risco
3,Paciente relata mal-estar passageiro esporádic...,baixo risco
4,Paciente apresenta mal-estar passageiro ocasio...,baixo risco
5,"Paciente relata pressão forte no peito, falta ...",alto risco
6,Paciente relata cansaço após subir escadas lev...,baixo risco
7,Paciente sente palpitações intensas acompanhad...,alto risco
8,Paciente sente palpitações intensas acompanhad...,alto risco
9,Paciente percebe sensação de coração acelerado...,baixo risco


### 2. Separação treino/teste
Usamos divisão estratificada para manter a proporção das classes nos dois conjuntos.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    df['frase'],
    df['rotulo'],
    test_size=0.25,
    random_state=42,
    stratify=df['rotulo']
)

print('Treino:', len(X_train))
print('Teste :', len(X_test))

Treino: 90
Teste : 30


### 3. TF-IDF
O TF-IDF transforma os textos em vetores numéricos, atribuindo maior importância a termos relevantes para diferenciar as frases.

In [4]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=1
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print('Matriz de treino:', X_train_tfidf.shape)
print('Total de termos/bigramas:', len(vectorizer.get_feature_names_out()))

Matriz de treino: (90, 362)
Total de termos/bigramas: 362


### 4. Treinamento do classificador
Foi escolhida Regressão Logística porque funciona bem com representações textuais esparsas como TF-IDF e mantém a solução simples e interpretável para o escopo do trabalho.

In [5]:
modelo = LogisticRegression(max_iter=1000, random_state=42)
modelo.fit(X_train_tfidf, y_train)

y_pred = modelo.predict(X_test_tfidf)

### 5. Avaliação

In [6]:
acuracia = accuracy_score(y_test, y_pred)
print(f'Acurácia: {acuracia:.2%}')
print('\nMatriz de confusão:')
print(confusion_matrix(y_test, y_pred, labels=['baixo risco', 'alto risco']))
print('\nRelatório de classificação:')
print(classification_report(y_test, y_pred, digits=3))

Acurácia: 100.00%

Matriz de confusão:
[[15  0]
 [ 0 15]]

Relatório de classificação:
              precision    recall  f1-score   support

  alto risco      1.000     1.000     1.000        15
 baixo risco      1.000     1.000     1.000        15

    accuracy                          1.000        30
   macro avg      1.000     1.000     1.000        30
weighted avg      1.000     1.000     1.000        30



### 6. Exemplos de previsão

In [7]:
exemplos = [
    'Paciente sente dor forte no peito com suor frio e falta de ar.',
    'Paciente relata cansaço leve depois de caminhar e melhora rapidamente em repouso.',
    'Paciente teve desmaio associado a palpitações intensas.'
]
exemplos_tfidf = vectorizer.transform(exemplos)
previsoes = modelo.predict(exemplos_tfidf)

for texto, pred in zip(exemplos, previsoes):
    print(f'{pred:11s} -> {texto}')

alto risco  -> Paciente sente dor forte no peito com suor frio e falta de ar.
baixo risco -> Paciente relata cansaço leve depois de caminhar e melhora rapidamente em repouso.
alto risco  -> Paciente teve desmaio associado a palpitações intensas.


### 7. Padrões, distorções e limitações

- O conjunto é **simulado, pequeno e balanceado**, portanto a acurácia obtida tende a ser mais alta do que em dados clínicos reais.
- Termos como *dor intensa*, *suor frio*, *desmaio* e *falta de ar em repouso* aparecem com maior frequência nos exemplos de alto risco; o modelo pode aprender esses padrões lexicais em vez de raciocínio clínico.
- Os exemplos não representam diversidade demográfica, comorbidades, erros de digitação, regionalismos ou formas reais de relato dos pacientes.
- Por isso, o resultado demonstra o funcionamento de uma pipeline de NLP/classificação, mas **não valida uso clínico**.
- Em uma aplicação real seriam necessários dados representativos, validação médica, métricas adicionais e análise formal de vieses antes de qualquer uso assistencial.

### Conclusão
A pipeline TF-IDF + Regressão Logística atende ao objetivo acadêmico de transformar frases médicas em vetores, treinar um classificador supervisionado e avaliar seu desempenho em exemplos não vistos.